In [1]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.1.1+cu118 requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.
torchtext 0.16.1+cpu requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.
torchvision 0.16.1+cu118 requires torch==2.1.1, but you have torch 2.10.0 which is incompatible.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 6454
LORA_RANK = 8

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

enable_thinking = False

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [3]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [ ]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_json("../data/preprocessed/train.jsonl", lines=True)
test_df  = pd.read_json("../data/preprocessed/test.jsonl",  lines=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()

In [ ]:
def format_chat(row):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["ideal_response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=enable_thinking)


train_df["text"] = train_df.apply(format_chat, axis=1)
dataset = Dataset.from_pandas(train_df[["text"]])

print(f"Example length (chars): {len(dataset[0]['text'])}")
print(dataset[0]["text"][:500])

In [ ]:
lengths = [len(tokenizer.encode(t)) for t in train_df["text"]]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)//len(lengths)}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"Примеров влезающих полностью: {sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)}/{len(lengths)}")

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        gradient_checkpointing=True,
    ),
)

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Adding EOS to train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


In [ ]:
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs[logs["loss"].notna()] if "loss" in logs.columns else logs

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], marker="o", color="#d62728")
axes[0].set_title("Training loss")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.3)

if "learning_rate" in train_logs.columns:
    axes[1].plot(train_logs["step"], train_logs["learning_rate"], marker="o", color="#1f77b4")
    axes[1].set_title("Learning rate")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("lr")
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## Проверка

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

model.eval()
model.config.use_cache = True

test_prompt  = test_df["prompt"].iloc[0]
test_system  = test_df["system_prompt"].iloc[0]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt", enable_thinking=enable_thinking).to(model.device)
print(f"Total input tokens: {inputs['input_ids'].shape[1]}")

with torch.no_grad(), torch.autocast("cuda"):
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Eval: ROUGE-L, BERTScore, Perplexity

In [ ]:
import sys, contextlib
sys.path.append("..")
from src.eval.metrics import compute_perplexity_hf, compute_rouge_l, compute_bert_score


def eval_one(model, row, disable_lora: bool):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True,
        return_tensors="pt", enable_thinking=enable_thinking,
    ).to(model.device)

    ctx = model.disable_adapter() if disable_lora else contextlib.nullcontext()
    with ctx:
        with torch.no_grad(), torch.autocast("cuda"):
            output_ids = model.generate(**inputs, max_new_tokens=2048, temperature=0.3, do_sample=True)

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        hypothesis = tokenizer.decode(generated_ids, skip_special_tokens=True)
        reference = row["ideal_response"]

        full_ids = tokenizer.apply_chat_template(
            messages + [{"role": "assistant", "content": reference}],
            return_dict=True, return_tensors="pt", enable_thinking=enable_thinking,
        ).to(model.device)
        ppl = compute_perplexity_hf(model, full_ids["input_ids"], full_ids["attention_mask"])

    rouge = compute_rouge_l(hypothesis, reference)
    bert_f1 = compute_bert_score(hypothesis, reference)
    return {"perplexity": ppl, "rouge_l": rouge, "bert_score_f1": bert_f1, "hypothesis": hypothesis}


results = []
for idx, row in test_df.iterrows():
    print(f"\n--- Test sample {idx} ---")
    for variant, disable in [("baseline", True), ("finetuned", False)]:
        m = eval_one(model, row, disable_lora=disable)
        print(f"  [{variant:9s}] PPL={m['perplexity']:.2f} | ROUGE-L={m['rouge_l']:.4f} | BERTScore={m['bert_score_f1']:.4f}")
        results.append({"sample": idx, "variant": variant, **{k: v for k, v in m.items() if k != "hypothesis"}})

eval_results = pd.DataFrame(results)
eval_results.to_csv("eval_baseline_vs_finetuned.csv", index=False)

summary = eval_results.groupby("variant")[["perplexity", "rouge_l", "bert_score_f1"]].mean()
print("\n=== Mean metrics ===")
print(summary.to_string())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ["perplexity", "rouge_l", "bert_score_f1"]
titles  = ["Perplexity ↓", "ROUGE-L ↑", "BERTScore F1 ↑"]
colors  = {"baseline": "#999999", "finetuned": "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, metric, title in zip(axes, metrics, titles):
    pivot = eval_results.pivot(index="sample", columns="variant", values=metric)
    x = np.arange(len(pivot))
    w = 0.38
    ax.bar(x - w/2, pivot["baseline"],  w, label="baseline",  color=colors["baseline"])
    ax.bar(x + w/2, pivot["finetuned"], w, label="finetuned", color=colors["finetuned"])
    ax.set_xticks(x); ax.set_xticklabels(pivot.index, rotation=0)
    ax.set_title(title); ax.set_xlabel("test sample")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig("eval_per_sample.png", dpi=120, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
means = eval_results.groupby("variant")[metrics].mean().reindex(["baseline", "finetuned"])
norm = means / means.loc["baseline"]
x = np.arange(len(metrics)); w = 0.38
ax.bar(x - w/2, norm.loc["baseline"],  w, label="baseline",  color=colors["baseline"])
ax.bar(x + w/2, norm.loc["finetuned"], w, label="finetuned", color=colors["finetuned"])
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(titles)
ax.set_title("Средние метрики (нормированы к baseline)")
ax.set_ylabel("отношение к baseline")
ax.grid(axis="y", alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig("eval_summary.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nГрафики сохранены: training_curves.png, eval_per_sample.png, eval_summary.png")

## Сохранение

Сохраняем LoRA-адаптер и (опционально) merged-модель в GGUF для Ollama.

In [ ]:
model.save_pretrained("koyash-qwen3-4b-lora")
tokenizer.save_pretrained("koyash-qwen3-4b-lora")

In [ ]:
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "koyash-qwen3-4b-lora",
    torch_dtype=torch.float16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("koyash-qwen3-4b-merged")
tokenizer.save_pretrained("koyash-qwen3-4b-merged")

## Конвертация в GGUF и загрузка в Ollama

```bash
# Клонируем llama.cpp и конвертируем
git clone https://github.com/ggerganov/llama.cpp
pip install -r llama.cpp/requirements.txt

python llama.cpp/convert_hf_to_gguf.py koyash-qwen3-4b-merged --outfile koyash-qwen3-4b.gguf --outtype q4_k_m

# Создаём Modelfile и загружаем в Ollama
echo 'FROM ./koyash-qwen3-4b.gguf' > Modelfile
ollama create koyash-qwen3-4b-lora -f Modelfile
ollama run koyash-qwen3-4b-lora
```